In [ ]:
%matplotlib inline
from pylab import *
from IPython.display import Image, display, HTML
import matplotlib.pyplot as plt

import math

In [ ]:
import pyagrum as gum
import pyagrum.lib.notebook as gnb
from pyagrum.lib import image as gumimage

In [ ]:
diag = gum.InfluenceDiagram()

O = diag.addChanceNode(gum.LabelizedVariable("O", "O", ["peach", "lemon"]))
R = diag.addChanceNode(gum.LabelizedVariable("R", "R", ["peach", "lemon"]))

T = diag.addDecisionNode(gum.LabelizedVariable("T", "T", ["yes", "no"]))
A = diag.addDecisionNode(gum.LabelizedVariable("A", "A", ["buy without guarantee", "buy with guarantee", "do not buy"]))

V1 = diag.addUtilityNode(gum.LabelizedVariable("V1", "V1", 1))  # cost of test
V2 = diag.addUtilityNode(gum.LabelizedVariable("V2", "V2", 1))  # gain when buying the car
V3 = diag.addUtilityNode(gum.LabelizedVariable("V3", "V3", 1))  # repairing costs


diag.addArc(T, R)
diag.addArc(O, R)
diag.addArc(R, A)
diag.addArc(T, V1)
diag.addArc(A, V2)
diag.addArc(A, V3)
diag.addArc(O, V3)


# diag.cpt(O).fillWith([0.8, 0.2])
# diag.cpt(R)[{"O": "peach", "T": "yes"}] = 0.8
# diag.cpt(R)[{"O": "lemon", "T": "yes"}] = 0.2
# diag.cpt(R)[{"O": "peach", "T": "no"}] = 0.5
# diag.cpt(R)[{"O": "lemon", "T": "no"}] = 0.5

# diag.utility(V1)[{"T": 0}] = -9
# diag.utility(V1)[{"T": 1}] = 0

# diag.utility(V2)[{"A": "buy without guarantee"}] = 100
# diag.utility(V2)[{"A": "buy with guarantee"}] = 40
# diag.utility(V2)[{"A": "do not buy"}] = 0

# diag.utility(V3)[{"A": "buy without guarantee", "O": "peach"}] = -40
# diag.utility(V3)[{"A": "buy without guarantee", "O": "lemon"}] = -200
# diag.utility(V3)[{"A": "buy with guarantee", "O": "peach"}] = -20
# diag.utility(V3)[{"A": "buy with guarantee", "O": "lemon"}] = 0
# diag.utility(V3)[{"A": "do not buy", "O": "peach"}] = 0
# diag.utility(V3)[{"A": "do not buy", "O": "lemon"}] = 0


# Values from DecisionProgramming
diag.cpt(O).fillWith([0.8, 0.2])
diag.cpt(R)[{"O": "peach", "T": "yes"}] = [1, 0]
diag.cpt(R)[{"O": "lemon", "T": "yes"}] = [0, 1]
diag.cpt(R)[{"O": "peach", "T": "no"}] = 0.5
diag.cpt(R)[{"O": "lemon", "T": "no"}] = 0.5

diag.utility(V1)[{"T": "yes"}] = -25
diag.utility(V1)[{"T": "no"}] = 0

diag.utility(V2)[{"A": "buy without guarantee"}] = 100
diag.utility(V2)[{"A": "buy with guarantee"}] = 40
diag.utility(V2)[{"A": "do not buy"}] = 0

diag.utility(V3)[{"A": "buy without guarantee", "O": "peach"}] = -40
diag.utility(V3)[{"A": "buy without guarantee", "O": "lemon"}] = -200
diag.utility(V3)[{"A": "buy with guarantee", "O": "peach"}] = -20
diag.utility(V3)[{"A": "buy with guarantee", "O": "lemon"}] = 0
diag.utility(V3)[{"A": "do not buy", "O": "peach"}] = 0
diag.utility(V3)[{"A": "do not buy", "O": "lemon"}] = 0



diag.saveBIFXML("bifxml/used_car_buyer_uniform.BIFXML")

In [ ]:
filename = f"figures/used_car_buyer_uniform.png"
gumimage.export(diag, filename)
display(Image(filename))


In [ ]:
gnb.flow.row(diag, gnb.getInference(diag))

In [ ]:
ie = gum.ShaferShenoyLIMIDInference(diag)
ie.makeInference()


In [ ]:
ie.posterior("T")

In [ ]:
ie.posterior("A")

In [ ]:
# a function to show results on decision nodes T and D
def show_decisions(ie):
  gnb.flow.row(
    ie.optimalDecision("T"),
    ie.optimalDecision("A"),
    f"{ie.MEU()['mean']:5.3f} (stdev : {math.sqrt(ie.MEU()['variance']):5.3f})",
    captions=["Strategy for T", "Strategy for A", "MEU and its standard deviation"],
  )
  gnb.flow.row(
    ie.posterior("T"),
    ie.posteriorUtility("T"),
    ie.posterior("A"),
    ie.posteriorUtility("A"),
    captions=[
      "Final decision for T",
      "Final reward for T",
      "Final decision for A",
      "Final reward for A",
    ],
  )


ie = gum.ShaferShenoyLIMIDInference(diag)

display(HTML("<h2>Inference in the LIMID optimizing the decisions nodes</h2>"))
ie.makeInference()
show_decisions(ie)

In [ ]:
ie.optimalDecision("A").names

In [ ]:
ie.optimalDecision("A").nbrDim()

In [ ]:
ie.optimalDecision("A").toarray()

In [ ]:
df = ie.optimalDecision("A").topandas()
df

In [ ]:
prob = ie.posterior("A")
util = ie.posteriorUtility("A")

In [ ]:
(prob.toarray() * util.toarray()).sum()

In [ ]:
prob.topandas()

In [ ]:
util.topandas()

In [ ]:
def copy(src):
    dst = gum.InfluenceDiagram()

    O = dst.addChanceNode(gum.LabelizedVariable("O", "O", ["peach", "lemon"]))
    R = dst.addChanceNode(gum.LabelizedVariable("R", "R", ["peach", "lemon"]))

    T = dst.addDecisionNode(gum.LabelizedVariable("T", "T", ["yes", "no"]))
    A = dst.addDecisionNode(gum.LabelizedVariable("A", "A", ["buy without guarantee", "buy with guarantee", "do not buy"]))

    V1 = dst.addUtilityNode(gum.LabelizedVariable("V1", "V1", 1))  # cost of test
    V2 = dst.addUtilityNode(gum.LabelizedVariable("V2", "V2", 1))  # gain when buying the car
    V3 = dst.addUtilityNode(gum.LabelizedVariable("V3", "V3", 1))  # repairing costs


    dst.addArc(T, R)
    dst.addArc(O, R)
    dst.addArc(R, A)
    dst.addArc(T, V1)
    dst.addArc(A, V2)
    dst.addArc(A, V3)
    dst.addArc(O, V3)

    dst.cpt(O).fillWith([0.8, 0.2])
    dst.cpt(R)[{"O": "peach", "T": "yes"}] = [1, 0]
    dst.cpt(R)[{"O": "lemon", "T": "yes"}] = [0, 1]
    dst.cpt(R)[{"O": "peach", "T": "no"}] = 0.5
    dst.cpt(R)[{"O": "lemon", "T": "no"}] = 0.5

    dst.utility(V1)[{"T": "yes"}] = -25
    dst.utility(V1)[{"T": "no"}] = 0

    dst.utility(V2)[{"A": "buy without guarantee"}] = 100
    dst.utility(V2)[{"A": "buy with guarantee"}] = 40
    dst.utility(V2)[{"A": "do not buy"}] = 0

    dst.utility(V3)[{"A": "buy without guarantee", "O": "peach"}] = -40
    dst.utility(V3)[{"A": "buy without guarantee", "O": "lemon"}] = -200
    dst.utility(V3)[{"A": "buy with guarantee", "O": "peach"}] = -20
    dst.utility(V3)[{"A": "buy with guarantee", "O": "lemon"}] = 0
    dst.utility(V3)[{"A": "do not buy", "O": "peach"}] = 0
    dst.utility(V3)[{"A": "do not buy", "O": "lemon"}] = 0

    return dst



def sa_testing(diag, min: float, max: float, prec: int):
    sa_diag = copy(diag)
    rng = np.linspace(min, max, num=prec, endpoint=True)
    count = rng.size
    meu = np.empty((count,))
    for k,v in enumerate(rng):
        sa_diag.cpt(R)[{"O": "peach", "T": "yes"}] = [v, 1-v]
        sa_diag.cpt(R)[{"O": "lemon", "T": "yes"}] = [1-v, v]
        
        # print(k, v)
        # print(sa_diag.cpt(R).toarray())
        # print(diag.cpt(R).toarray())
        ie_ = gum.ShaferShenoyLIMIDInference(sa_diag)
        ie_.makeInference()
        prob = ie_.posterior("A").toarray()
        util = ie_.posteriorUtility("A").toarray()

        meu[k] = (prob * util).sum()
        # print(meu[k])
        # print()

    return meu


In [ ]:
%%timeit
dist_meu = sa_testing(diag, 0, 1, 1001)

In [ ]:
dist_meu = sa_testing(diag, 0, 1, 1001)

In [ ]:
dist_meu[0]

In [ ]:
plt.hist(dist_meu)